# Neural activity overview and Peterka-inspired V1 analysis

This notebook first checks the complete denoised calcium matrix and atlas, then tests the train and test predictions motivated by Peterka et al. (2026), *Global context rapidly shapes sensory responses in V1*.

The paper compared every P5 orientation with the same orientation in a random-sequence control. This experiment has no random-sequence control, so the analyses below are explicitly **within-paradigm proxies**, not a replication of deviance detection. The same fixed V1 neuron population is retained across conditions and test P5-A/B/C trial counts are balanced before plotting or statistics.

Source data are read-only. Derived results are saved under `outputs/<session>/neural_analysis/peterka_inspired/`.

In [ ]:
# Block N1 - imports and plotting defaults
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from attention_alignment import load_config
from attention_alignment.neural_analysis import (
    adjust_pvalues_holm,
    analyze_balanced_p5_responses,
    analyze_train_sequence_adaptation,
    build_sequence_trace_windows,
    build_stimulus_windows,
    compare_neuron_response_means,
    compute_paired_p5_neuron_effects,
    compute_peak_time_row_order,
    compute_stimulus_response_statistics,
    downsample_neural_activity,
    estimate_baseline_noise_scale,
    extract_event_aligned_neural_traces,
    load_neuron_atlas,
    open_neural_activity,
    paired_equivalence_test,
    plot_activity_heatmap,
    plot_balanced_p5_heatmaps,
    plot_balanced_p5_traces,
    plot_fixed_population_condition_atlas,
    plot_neurons_on_atlas,
    plot_p5_neuron_effect_scatter,
    plot_representative_p5_traces,
    plot_train_adaptation_summary,
    plot_train_sequence_mean_sem,
    response_delta_from_statistics,
    select_atlas_cell_indices,
    select_balanced_test_p5_events,
    select_paired_catch_reference_events,
    select_representative_p5_neurons,
    subset_stimulus_response_statistics,
    summarize_mean_stimulus_responses,
    summarize_stimulus_responses,
)
from attention_alignment.pupil_analysis import save_figure_bundle

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 10,
})

def display_figure_once(figure):
    display(figure)
    plt.close(figure)

CONFIG_PATH = Path('configs/sessions.local.yaml')
CONFIG = load_config(CONFIG_PATH)

In [ ]:
# Block N2 - user-editable parameters
SESSION_ID = 'replace_with_session_id'
TRACE_RELATIVE_PATH = Path('denoise') / 'whole_brain_trace_denoised_loose.npy'
ATLAS_FILE = 'valid_area_5.mat'

# Visual-cortex population: match all Allen acronyms beginning with VIS.
ANALYSIS_REGION = 'VIS'

# Every interval is converted with real aligned timestamps, never a fixed frame rate.
BASELINE_WINDOW_S = (-2.5, -0.5)
RESPONSE_WINDOW_S = (0.0, None)  # None means the measured stimulus-off timestamp.
DISPLAY_PRE_STIMULUS_S = 0.5
POST_STIMULUS_DISPLAY_S = 0.5
MIN_WINDOW_FRAMES = 2
PROCESSING_CHUNK_SIZE = 128
BASELINE_SCALE_FLOOR_PERCENTILE = 5.0

# Main Figure-S1-style population is P1-A-responsive OR P5-B-responsive VISp.
# This experiment retains first/last 10; trajectory binning is separate.
TRAIN_EDGE_TRIALS = 10
TRAIN_BIN_SIZE = 10  # Change to 5 or 10 to alter the trajectory grouping.
PETERKA_RESPONSE_REGION = 'VISp'
PETERKA_ORIENTATION_RESPONSE_SD_THRESHOLD = 1.97
PETERKA_ORIENTATION_MIN_TRIALS = 4

# P5 A/B/C are balanced to the smallest available count (currently 10 each).
TEST_BALANCED_TRIALS = None
RANDOM_SEED = 2026
B_EQUIVALENCE_MARGIN_SD = 0.20

# Exploratory condition-responsive screen, retained from the reference analysis.
RESPONSIVE_STD_MULTIPLIER = 1.0
REQUIRE_RESPONSE_STD_GT_BASELINE = True
MIN_RESPONSIVE_TRIALS = 4
MIN_RESPONSIVE_TRIAL_FRACTION = 0.50

HEATMAP_MAX_TIME_BINS = 2000
HEATMAP_DOWNSAMPLE_METHOD = 'sample'  # 'mean' is slower and averages all frames.
HEATMAP_AUTO_PERCENTILES = (1.0, 99.0)
HEATMAP_CMAP = 'viridis'
MATCHED_HEATMAP_ROBUST_PERCENTILE = 99.0
EFFECT_SCATTER_AXIS_PERCENTILE = 99.5
REPRESENTATIVE_NEURONS_PER_GROUP = 2
REPRESENTATIVE_MIN_VALID_FRACTION = 0.8
SAVE_RESULTS = True

SESSION_SOURCE_DIR = CONFIG.session_dir(SESSION_ID)
SESSION_OUTPUT_DIR = CONFIG.output_root / SESSION_ID
ANALYSIS_DIR = SESSION_OUTPUT_DIR / 'neural_analysis' / 'peterka_inspired_VIS_all'
FIGURE_DIR = ANALYSIS_DIR / 'figures'

print({
    'session': SESSION_ID,
    'analysis_region': ANALYSIS_REGION,
    'train_first_last_n': TRAIN_EDGE_TRIALS,
    'train_bin_size': TRAIN_BIN_SIZE,
    'train_response_region': PETERKA_RESPONSE_REGION,
    'orientation_response_sd': PETERKA_ORIENTATION_RESPONSE_SD_THRESHOLD,
    'response_window_s': RESPONSE_WINDOW_S,
    'output': ANALYSIS_DIR,
})

In [ ]:
# Block N3 - load and cross-check aligned frames, activity, events, and atlas
frame_path = SESSION_OUTPUT_DIR / 'calcium_frames.parquet'
event_path = SESSION_OUTPUT_DIR / 'stimulus_events.parquet'
trace_path = SESSION_SOURCE_DIR / TRACE_RELATIVE_PATH
atlas_path = SESSION_SOURCE_DIR / ATLAS_FILE

calcium_frames = pd.read_parquet(frame_path).sort_values('hdf5_frame_index').reset_index(drop=True)
stimulus_events = pd.read_parquet(event_path).reset_index(drop=True)
stimulus_events['source_event_index'] = np.arange(len(stimulus_events), dtype=np.int64)
atlas, ccf = load_neuron_atlas(atlas_path)

expected_frame_indices = np.arange(len(calcium_frames), dtype=np.int64)
actual_frame_indices = calcium_frames['hdf5_frame_index'].to_numpy(dtype=np.int64)
if not np.array_equal(actual_frame_indices, expected_frame_indices):
    raise RuntimeError('calcium_frames must be contiguous and ordered before neural analysis')

frame_times_s = calcium_frames['t_session_s'].to_numpy(dtype=float)
activity = open_neural_activity(
    trace_path, frame_count=len(calcium_frames), cell_count=len(atlas)
)
median_frame_interval_s = float(np.median(np.diff(frame_times_s)))
data_qc = pd.DataFrame([{
    'session_id': SESSION_ID,
    'activity_shape_cells_by_frames': str(activity.shape),
    'atlas_neurons': len(atlas),
    'aligned_frames': len(calcium_frames),
    'measured_frame_rate_hz': 1.0 / median_frame_interval_s,
    'duration_min': (frame_times_s[-1] - frame_times_s[0]) / 60.0,
    'trace_dtype': str(activity.dtype),
    'trace_is_memory_mapped': bool(
        isinstance(activity, np.memmap) or isinstance(activity.base, np.memmap)
    ),
}])
display(data_qc)
display(stimulus_events.groupby(['phase', 'item_position', 'symbol']).size().rename('events').reset_index())

In [ ]:
# Block N4 - complete-recording activity and all-neuron atlas overview
heatmap_times_s, activity_heatmap, heatmap_cell_indices = downsample_neural_activity(
    activity, frame_times_s,
    max_time_bins=HEATMAP_MAX_TIME_BINS,
    chunk_size=PROCESSING_CHUNK_SIZE,
    method=HEATMAP_DOWNSAMPLE_METHOD,
)
heatmap_vmin, heatmap_vmax = np.nanpercentile(
    activity_heatmap, HEATMAP_AUTO_PERCENTILES
)
fig_activity_overview = plot_activity_heatmap(
    activity_heatmap, heatmap_times_s,
    vmin=heatmap_vmin, vmax=heatmap_vmax, cmap=HEATMAP_CMAP,
    title='Whole-brain denoised activity',
)
fig_all_neurons_atlas = plot_neurons_on_atlas(
    atlas, ccf, title=f'{SESSION_ID}: all detected neurons (n={len(atlas):,})'
)
display_figure_once(fig_activity_overview)
display_figure_once(fig_all_neurons_atlas)
print({
    'source_shape': activity.shape,
    'display_shape': activity_heatmap.shape,
    'display_only_downsampling': True,
    'color_limits': (float(heatmap_vmin), float(heatmap_vmax)),
})

In [ ]:
# Block N5 - define the fixed analysis population
analysis_cell_indices = atlas.loc[
    atlas['atlas_acronym'].fillna('').astype(str).str.startswith(ANALYSIS_REGION),
    'cell_index',
].to_numpy(dtype=np.int64)
if len(analysis_cell_indices) == 0:
    raise ValueError(f'No atlas neurons match prefix {ANALYSIS_REGION!r}')
analysis_atlas = atlas.loc[
    atlas['cell_index'].isin(analysis_cell_indices)
].copy().sort_values('cell_index')
if not np.array_equal(
    analysis_atlas['cell_index'].to_numpy(dtype=np.int64), analysis_cell_indices
):
    raise RuntimeError('Atlas and activity cell ordering differ for the analysis population')

analysis_population_qc = pd.DataFrame([{
    'region': ANALYSIS_REGION if ANALYSIS_REGION is not None else 'all',
    'fixed_neuron_count': len(analysis_cell_indices),
    'total_neuron_count': len(atlas),
    'fraction_of_recording': len(analysis_cell_indices) / len(atlas),
    'random_sequence_control_available': False,
    'biological_replicates_in_this_notebook': 1,
}])
display(analysis_population_qc)
display(analysis_atlas['atlas_acronym'].value_counts().rename('neurons').to_frame())
print(
    'Interpretation boundary: all tests below are within one mouse and lack the '
    'same-orientation random-sequence control used by Peterka et al.'
)

In [ ]:
# Block N6 - one shared timestamp-aligned response pass for train and test
train_windows = build_stimulus_windows(
    stimulus_events, frame_times_s,
    phase='train', symbols=None, item_positions=(1, 2, 3, 4, 5),
    baseline_window_s=BASELINE_WINDOW_S,
    response_window_s=RESPONSE_WINDOW_S,
    min_window_frames=MIN_WINDOW_FRAMES,
)
train_statistics = compute_stimulus_response_statistics(
    activity, train_windows,
    cell_indices=analysis_cell_indices,
    min_valid_frames=MIN_WINDOW_FRAMES,
    chunk_size=PROCESSING_CHUNK_SIZE,
)
neuron_scale = estimate_baseline_noise_scale(
    train_statistics, floor_percentile=BASELINE_SCALE_FLOOR_PERCENTILE
)
train_analysis = analyze_train_sequence_adaptation(
    train_statistics, train_windows, scale=neuron_scale,
    early_trial_count=TRAIN_EDGE_TRIALS,
    late_trial_count=TRAIN_EDGE_TRIALS,
    bin_size=TRAIN_BIN_SIZE,
)

# Test P4 is retained for the same-orientation A proxy; P5 supplies A/B/C.
test_windows = build_stimulus_windows(
    stimulus_events, frame_times_s,
    phase='test', symbols=None, item_positions=(4, 5),
    baseline_window_s=BASELINE_WINDOW_S,
    response_window_s=RESPONSE_WINDOW_S,
    min_window_frames=MIN_WINDOW_FRAMES,
)
test_statistics = compute_stimulus_response_statistics(
    activity, test_windows,
    cell_indices=analysis_cell_indices,
    min_valid_frames=MIN_WINDOW_FRAMES,
    chunk_size=PROCESSING_CHUNK_SIZE,
)
balanced_p5_events = select_balanced_test_p5_events(
    stimulus_events,
    trial_count=TEST_BALANCED_TRIALS,
    random_seed=RANDOM_SEED,
)
balance_annotations = balanced_p5_events.set_index('source_event_index')[
    ['balance_source', 'balanced_trial_index']
]
for column in balance_annotations.columns:
    test_windows[column] = test_windows['source_event_index'].map(
        balance_annotations[column]
    )
test_metadata = test_windows.loc[
    np.asarray(test_statistics['window_index'], dtype=np.int64)
].reset_index()
balanced_sources = set(
    balanced_p5_events['source_event_index'].to_numpy(dtype=np.int64)
)
balanced_mask = test_metadata['source_event_index'].isin(balanced_sources).to_numpy()
balanced_statistics = subset_stimulus_response_statistics(
    test_statistics, balanced_mask
)
balanced_windows = test_windows.loc[
    np.asarray(balanced_statistics['window_index'], dtype=np.int64)
]
balanced_metadata = balanced_windows.reset_index()
test_analysis = analyze_balanced_p5_responses(
    balanced_statistics, test_windows, scale=neuron_scale
)

shared_pass_qc = pd.DataFrame([
    {
        'phase': 'train',
        'matching_events': len(train_windows),
        'valid_events': int(train_windows['window_valid'].sum()),
        'neurons': len(analysis_cell_indices),
    },
    {
        'phase': 'test P4/P5',
        'matching_events': len(test_windows),
        'valid_events': int(test_windows['window_valid'].sum()),
        'neurons': len(analysis_cell_indices),
    },
    {
        'phase': 'balanced test P5',
        'matching_events': len(balanced_windows),
        'valid_events': int(balanced_windows['window_valid'].sum()),
        'neurons': len(analysis_cell_indices),
    },
])
display(shared_pass_qc)
display(
    balanced_metadata.groupby(['symbol', 'balance_source']).size()
    .rename('trials').reset_index()
)

In [ ]:
# Block N7 - train: full-sequence Peterka-style populations
train_statistics_metadata = train_windows.loc[
    np.asarray(train_statistics['window_index'], dtype=np.int64)
].reset_index()
train_p1_a_mask = (
    train_statistics_metadata['item_position'].eq(1)
    & train_statistics_metadata['symbol'].eq('A')
).to_numpy()
train_p5_b_mask = (
    train_statistics_metadata['item_position'].eq(5)
    & train_statistics_metadata['symbol'].eq('B')
).to_numpy()
train_p1_a_statistics = subset_stimulus_response_statistics(
    train_statistics, train_p1_a_mask
)
train_p5_b_statistics = subset_stimulus_response_statistics(
    train_statistics, train_p5_b_mask
)
train_p1_a_mean_selection = summarize_mean_stimulus_responses(
    train_p1_a_statistics,
    std_multiplier=PETERKA_ORIENTATION_RESPONSE_SD_THRESHOLD,
    min_eligible_trials=PETERKA_ORIENTATION_MIN_TRIALS,
)
train_p5_b_mean_selection = summarize_mean_stimulus_responses(
    train_p5_b_statistics,
    std_multiplier=PETERKA_ORIENTATION_RESPONSE_SD_THRESHOLD,
    min_eligible_trials=PETERKA_ORIENTATION_MIN_TRIALS,
)
peterka_candidate_cells = select_atlas_cell_indices(
    atlas, PETERKA_RESPONSE_REGION
)
train_ab_mean_selection = train_p1_a_mean_selection.merge(
    train_p5_b_mean_selection, on='cell_index', suffixes=('_A', '_B'),
    validate='one_to_one',
)
train_ab_mean_selection['selected'] = (
    train_ab_mean_selection['selected_A']
    | train_ab_mean_selection['selected_B']
)
train_ab_neuron_selection = atlas.merge(
    train_ab_mean_selection, on='cell_index', validate='one_to_one'
).loc[lambda table: table['cell_index'].isin(peterka_candidate_cells)].copy()
train_ab_selected = train_ab_neuron_selection.loc[
    train_ab_neuron_selection['selected']
].copy()
train_p5_b_neuron_selection = atlas.merge(
    train_p5_b_mean_selection, on='cell_index', validate='one_to_one'
).loc[lambda table: table['cell_index'].isin(peterka_candidate_cells)].copy()
train_p5_b_selected = train_p5_b_neuron_selection.loc[
    train_p5_b_neuron_selection['selected']
].copy()
if train_ab_selected.empty or train_p5_b_selected.empty:
    raise ValueError(
        'No orientation-responsive VISp neurons passed the Train A/B mean-response '
        'screen. Review PETERKA_ORIENTATION_RESPONSE_SD_THRESHOLD.'
    )
analysis_row_by_cell = pd.Series(
    np.arange(len(analysis_cell_indices), dtype=np.int64),
    index=analysis_cell_indices,
)
train_ab_selected_rows = analysis_row_by_cell.loc[
    train_ab_selected['cell_index'].to_numpy(dtype=np.int64)
].to_numpy(dtype=np.int64)
def subset_train_statistics(row_indices, cell_indices):
    result = {
        key: np.asarray(train_statistics[key])[row_indices]
        for key in ('baseline_mean', 'baseline_std', 'response_mean', 'response_std', 'eligible')
    }
    result['cell_index'] = np.asarray(cell_indices, dtype=np.int64)
    result['window_index'] = np.asarray(train_statistics['window_index'], dtype=np.int64)
    return result

peterka_neuron_scale = neuron_scale[train_ab_selected_rows]
peterka_train_statistics = subset_train_statistics(
    train_ab_selected_rows, train_ab_selected['cell_index']
)
peterka_train_analysis = analyze_train_sequence_adaptation(
    peterka_train_statistics, train_windows, scale=peterka_neuron_scale,
    early_trial_count=TRAIN_EDGE_TRIALS,
    late_trial_count=TRAIN_EDGE_TRIALS,
    bin_size=TRAIN_BIN_SIZE,
)
edge_sequences = (
    set(peterka_train_analysis['early_sequences'])
    | set(peterka_train_analysis['late_sequences'])
)
late_train_p5_windows = train_windows.loc[
    train_windows['window_valid']
    & train_windows['sequence_index'].isin(
        peterka_train_analysis['late_sequences']
    )
    & train_windows['item_position'].eq(5)
    & train_windows['symbol'].eq('B')
]
train_stimulus_duration_s = float(np.nanmedian(
    late_train_p5_windows['measured_offset_s']
    - late_train_p5_windows['measured_onset_s']
))
train_sort_reference_trace_window_s = (
    float(BASELINE_WINDOW_S[0]),
    train_stimulus_duration_s + POST_STIMULUS_DISPLAY_S,
)
train_sort_reference_aligned = extract_event_aligned_neural_traces(
    activity, frame_times_s, late_train_p5_windows, analysis_cell_indices,
    trace_window_s=train_sort_reference_trace_window_s,
    baseline_window_s=BASELINE_WINDOW_S,
)
train_sort_reference_delta_std = np.divide(
    train_sort_reference_aligned['delta'], neuron_scale[:, None, None],
    out=np.full_like(train_sort_reference_aligned['delta'], np.nan),
    where=np.isfinite(neuron_scale[:, None, None]) & (neuron_scale[:, None, None] > 0),
)
sequence_data = build_sequence_trace_windows(
    train_windows, sequence_indices=sorted(edge_sequences)
)
train_sequence_windows = sequence_data['sequence_windows']
train_sequence_stimulus_timing = sequence_data['stimulus_timing']
train_sequence_trace_window_s = (
    float(BASELINE_WINDOW_S[0]),
    float(train_sequence_windows['sequence_duration_s'].max())
    + POST_STIMULUS_DISPLAY_S,
)
train_sequence_aligned = extract_event_aligned_neural_traces(
    activity, frame_times_s, train_sequence_windows,
    train_ab_selected['cell_index'].to_numpy(dtype=np.int64),
    trace_window_s=train_sequence_trace_window_s,
    baseline_window_s=BASELINE_WINDOW_S,
)
train_sequence_metadata = train_sequence_windows.loc[
    train_sequence_aligned['window_index']
].reset_index()
train_sequence_delta_raw = train_sequence_aligned['delta']
train_sequence_population_groups = {
    'A-only': (
        train_ab_selected['selected_A']
        & ~train_ab_selected['selected_B']
    ).to_numpy(dtype=bool),
    'B-only': (
        train_ab_selected['selected_B']
        & ~train_ab_selected['selected_A']
    ).to_numpy(dtype=bool),
    'A-or-B union': train_ab_selected['selected'].to_numpy(dtype=bool),
}
fig_train_sequence_mean_sem = plot_train_sequence_mean_sem(
    train_sequence_delta_raw, train_sequence_aligned['relative_time_s'],
    train_sequence_metadata, train_sequence_stimulus_timing,
    early_sequences=peterka_train_analysis['early_sequences'],
    late_sequences=peterka_train_analysis['late_sequences'],
    population_groups=train_sequence_population_groups,
    ylabel='Mean +/- SEM baseline-subtracted activity (a.u.)',
    title=(
        f'{SESSION_ID} | raw A-only, B-only, and union {PETERKA_RESPONSE_REGION}* '
        '| full AAAAB sequence (no baseline-SD scaling)'
    ),
)
fig_train_summary = plot_train_adaptation_summary(
    peterka_train_analysis,
    title=(
        f'{SESSION_ID} | A-or-B-responsive {PETERKA_RESPONSE_REGION}* '
        f'(n={len(train_ab_selected):,}) | first/last {TRAIN_EDGE_TRIALS}'
    ),
)
display(peterka_train_analysis['p5_directional_test'])
display(peterka_train_analysis['p5_spearman_test'])
display(peterka_train_analysis['position_tests'][[
    'item_position', 'symbol', 'n_neurons', 'mean_difference',
    'cohen_dz', 'p_value', 'p_holm_5_positions',
]])
display(pd.DataFrame([
    {
        'population': label,
        'candidate_neurons': len(train_p5_b_neuron_selection),
        'neurons': int(mask.sum()),
        'selection_data': {
            'A-only': 'P1-A screen only',
            'B-only': 'P5-B screen only',
            'A-or-B union': 'P1-A OR P5-B screen',
        }[label],
        'minimum_trial_fraction': 'not used',
        'response_threshold_baseline_sd': PETERKA_ORIENTATION_RESPONSE_SD_THRESHOLD,
    }
    for label, mask in train_sequence_population_groups.items()
]))
display_figure_once(fig_train_sequence_mean_sem)
display_figure_once(fig_train_summary)
print(
    'Main full-sequence rows show raw baseline-subtracted A-only, B-only, '
    'and union activity on one shared y-axis; no baseline-SD scaling is used.'
)

In [ ]:
# Block N8 - test: fixed-cell balanced A/B/C and literature-logic proxy contrasts
test_stimulus_duration_s = float(np.nanmedian(
    balanced_windows['measured_offset_s'] - balanced_windows['measured_onset_s']
))
test_trace_window_s = (
    float(BASELINE_WINDOW_S[0]),
    test_stimulus_duration_s + POST_STIMULUS_DISPLAY_S,
)
test_display_window_s = (
    -DISPLAY_PRE_STIMULUS_S,
    test_stimulus_duration_s + POST_STIMULUS_DISPLAY_S,
)
balanced_aligned = extract_event_aligned_neural_traces(
    activity, frame_times_s, balanced_windows, analysis_cell_indices,
    trace_window_s=test_trace_window_s,
    baseline_window_s=BASELINE_WINDOW_S,
)
balanced_aligned_metadata = test_windows.loc[
    balanced_aligned['window_index']
].reset_index()
balanced_aligned_delta_std = np.divide(
    balanced_aligned['delta'], neuron_scale[:, None, None],
    out=np.full_like(balanced_aligned['delta'], np.nan),
    where=np.isfinite(neuron_scale[:, None, None]) & (neuron_scale[:, None, None] > 0),
)
train_metadata = train_windows.loc[
    np.asarray(train_statistics['window_index'], dtype=np.int64)
].reset_index()
train_delta_std = response_delta_from_statistics(
    train_statistics, scale=neuron_scale
)
test_delta_std = response_delta_from_statistics(
    test_statistics, scale=neuron_scale
)
balanced_delta_std = response_delta_from_statistics(
    balanced_statistics, scale=neuron_scale
)

# Proxy 1: same orientation and same AAAAA sequence, but P5 vs P4 remains position-confounded.
aaaaa_p5 = (
    test_metadata['sequence_pattern'].eq('AAAAA')
    & test_metadata['item_position'].eq(5)
).to_numpy()
aaaaa_p4 = (
    test_metadata['sequence_pattern'].eq('AAAAA')
    & test_metadata['item_position'].eq(4)
).to_numpy()

# Proxy 2: same B orientation/AAAAB pattern, but test vs late train remains phase-confounded.
late_train_b = (
    train_metadata['sequence_index'].isin(train_analysis['late_sequences'])
    & train_metadata['item_position'].eq(5)
    & train_metadata['symbol'].eq('B')
).to_numpy()
balanced_b = balanced_metadata['symbol'].eq('B').to_numpy()

# Proxy 3: P5-C vs balanced P5-B retains the orientation identity confound.
balanced_c = balanced_metadata['symbol'].eq('C').to_numpy()

proxy_rows = [
    compare_neuron_response_means(
        test_delta_std[:, aaaaa_p5], test_delta_std[:, aaaaa_p4],
        comparison='P5-A < P4-A within AAAAA', alternative='less',
    ),
    compare_neuron_response_means(
        balanced_delta_std[:, balanced_b], train_delta_std[:, late_train_b],
        comparison='test P5-B vs late-train P5-B', alternative='two-sided',
    ),
    compare_neuron_response_means(
        balanced_delta_std[:, balanced_c], balanced_delta_std[:, balanced_b],
        comparison='P5-C > balanced P5-B', alternative='greater',
    ),
]
proxy_contrasts = pd.DataFrame(proxy_rows)
proxy_contrasts['p_holm_3_proxies'] = adjust_pvalues_holm(
    proxy_contrasts['p_value']
)
proxy_contrasts['interpretation_limit'] = [
    'same orientation; P5 vs P4 position confound',
    'same orientation/pattern; train-test phase confound',
    'matched P5 and trial count; B/C orientation confound',
]
b_equivalence = pd.DataFrame([{
    'comparison': (
        f'test P5-B equivalent to late-train P5-B within +/-'
        f'{B_EQUIVALENCE_MARGIN_SD:g} baseline SD'
    ),
    **paired_equivalence_test(
        balanced_delta_std[:, balanced_b],
        train_delta_std[:, late_train_b],
        margin=B_EQUIVALENCE_MARGIN_SD,
    ),
}])

display(test_analysis['condition_summary'])
display(test_analysis['omnibus_test'])
display(test_analysis['pairwise_tests'])
display(proxy_contrasts)
display(b_equivalence)
print(
    'A non-significant B difference is not evidence of no effect. '
    'Use the TOST row above for the pre-specified equivalence margin.'
)

In [ ]:
# Block N9 - matched A/B/C time courses, paired effects, and representative cells
late_train_p5_mean = np.nanmean(
    train_sort_reference_delta_std, axis=1
)
matched_row_sort = compute_peak_time_row_order(
    late_train_p5_mean,
    train_sort_reference_aligned['relative_time_s'],
    response_window_s=(0.0, train_stimulus_duration_s),
)

fig_test_p5_traces = plot_balanced_p5_traces(
    balanced_aligned_delta_std,
    balanced_aligned['relative_time_s'],
    balanced_aligned_metadata,
    stimulus_duration_s=test_stimulus_duration_s,
    display_window_s=test_display_window_s,
)
fig_test_p5_heatmaps, matched_heatmap_data = plot_balanced_p5_heatmaps(
    balanced_aligned_delta_std,
    balanced_aligned['relative_time_s'],
    balanced_aligned_metadata,
    stimulus_duration_s=test_stimulus_duration_s,
    row_order=matched_row_sort['row_order'],
    normalization='baseline_sd',
    z_limit=None,
    robust_percentile=MATCHED_HEATMAP_ROBUST_PERCENTILE,
    row_order_label='ordered by independent late-train P5-B peak time',
    display_window_s=test_display_window_s,
)
p5_paired_events = select_paired_catch_reference_events(
    stimulus_events, catch_symbols=('A', 'C'), reference_symbol='B'
)
p5_neuron_effects = compute_paired_p5_neuron_effects(
    test_delta_std, test_metadata, p5_paired_events,
    cell_indices=analysis_cell_indices,
).merge(
    analysis_atlas[['cell_index', 'atlas_id', 'atlas_acronym', 'atlas_name']],
    on='cell_index', how='left', validate='one_to_one',
)
representative_p5_neurons = select_representative_p5_neurons(
    p5_neuron_effects,
    per_group=REPRESENTATIVE_NEURONS_PER_GROUP,
    minimum_valid_fraction=REPRESENTATIVE_MIN_VALID_FRACTION,
)
fig_neuron_effect_scatter, neuron_effect_quadrants = plot_p5_neuron_effect_scatter(
    p5_neuron_effects, representatives=representative_p5_neurons,
    axis_percentile=EFFECT_SCATTER_AXIS_PERCENTILE,
)
fig_representative_traces = plot_representative_p5_traces(
    balanced_aligned_delta_std, balanced_aligned['relative_time_s'],
    balanced_aligned_metadata, analysis_cell_indices, representative_p5_neurons,
    stimulus_duration_s=test_stimulus_duration_s,
    display_window_s=test_display_window_s,
)

display(neuron_effect_quadrants)
display(representative_p5_neurons[[
    'representative_group', 'cell_index', 'atlas_acronym',
    'A_minus_B_A', 'C_minus_B_C', 'minimum_valid_fraction',
]])
display_figure_once(fig_test_p5_traces)
display_figure_once(fig_test_p5_heatmaps)
display_figure_once(fig_representative_traces)
display_figure_once(fig_neuron_effect_scatter)
print({
    'same_neurons_in_all_panels': len(analysis_cell_indices),
    'same_row_order_in_all_heatmaps': True,
    'row_order_reference': 'independent late-train P5-B peak time',
    'heatmap_normalization': 'baseline_sd',
    'balanced_trial_counts': balanced_aligned_metadata.groupby('symbol').size().to_dict(),
    'formal_deviant_minus_control_available': False,
})

In [ ]:
# Block N10 - independent responsiveness labels on the fixed V1 denominator
condition_selections = {}
selection_rows = []
for symbol in ('A', 'B', 'C'):
    condition_mask = balanced_metadata['symbol'].eq(symbol).to_numpy()
    condition_statistics = subset_stimulus_response_statistics(
        balanced_statistics, condition_mask
    )
    selection = summarize_stimulus_responses(
        condition_statistics,
        std_multiplier=RESPONSIVE_STD_MULTIPLIER,
        require_response_std_gt_baseline=REQUIRE_RESPONSE_STD_GT_BASELINE,
        min_responsive_trials=MIN_RESPONSIVE_TRIALS,
        min_responsive_fraction=MIN_RESPONSIVE_TRIAL_FRACTION,
    )
    condition_selections[symbol] = selection
    selection_rows.append({
        'condition': symbol,
        'fixed_denominator_neurons': len(selection),
        'selected_neurons': int(selection['selected'].sum()),
        'selected_fraction': float(selection['selected'].mean()),
        'trial_count': int(condition_mask.sum()),
    })
condition_selection_summary = pd.DataFrame(selection_rows)
fig_condition_atlas = plot_fixed_population_condition_atlas(
    atlas, ccf, condition_selections,
    title=f'P5-responsive subsets within fixed {ANALYSIS_REGION} population',
)
display(condition_selection_summary)
display_figure_once(fig_condition_atlas)
print(
    'These masks are independent descriptive screens. Counts must not be treated '
    'as a deviance-detection test or directly subtracted across A/B/C.'
)

In [ ]:
# Block N11 - save all reviewable tables, arrays, figures, and exact parameters
analysis_parameters = {
    'session_id': SESSION_ID,
    'trace_source': str(trace_path),
    'atlas_source': str(atlas_path),
    'analysis_region': ANALYSIS_REGION,
    'analysis_cell_count': int(len(analysis_cell_indices)),
    'baseline_window_s': list(BASELINE_WINDOW_S),
    'response_window_s': list(RESPONSE_WINDOW_S),
    'display_pre_stimulus_s': DISPLAY_PRE_STIMULUS_S,
    'post_stimulus_display_s': POST_STIMULUS_DISPLAY_S,
    'train_sort_reference_trace_window_s': list(train_sort_reference_trace_window_s),
    'train_sequence_trace_window_s': list(train_sequence_trace_window_s),
    'test_trace_window_s': list(test_trace_window_s),
    'minimum_window_frames': MIN_WINDOW_FRAMES,
    'baseline_scale_floor_percentile': BASELINE_SCALE_FLOOR_PERCENTILE,
    'train_edge_trials': TRAIN_EDGE_TRIALS,
    'train_bin_size': TRAIN_BIN_SIZE,
    'test_balanced_trials_requested': TEST_BALANCED_TRIALS,
    'test_balanced_trials_observed': (
        balanced_metadata.groupby('symbol').size().astype(int).to_dict()
    ),
    'random_seed': RANDOM_SEED,
    'matched_heatmap_normalization': 'baseline_sd',
    'matched_heatmap_robust_percentile': MATCHED_HEATMAP_ROBUST_PERCENTILE,
    'matched_heatmap_row_order_reference': 'independent late-train P5-B peak time',
    'effect_scatter_axis_percentile': EFFECT_SCATTER_AXIS_PERCENTILE,
    'paired_reference_definition': 'nearest_preceding_test_P5_B',
    'representative_neurons_per_group': REPRESENTATIVE_NEURONS_PER_GROUP,
    'representative_minimum_valid_fraction': REPRESENTATIVE_MIN_VALID_FRACTION,
    'b_equivalence_margin_baseline_sd': B_EQUIVALENCE_MARGIN_SD,
    'responsive_std_multiplier': RESPONSIVE_STD_MULTIPLIER,
    'require_response_std_gt_baseline': REQUIRE_RESPONSE_STD_GT_BASELINE,
    'minimum_responsive_trials': MIN_RESPONSIVE_TRIALS,
    'minimum_responsive_trial_fraction': MIN_RESPONSIVE_TRIAL_FRACTION,
    'train_orientation_response_region': PETERKA_RESPONSE_REGION,
    'train_orientation_response_sd_threshold': (
        PETERKA_ORIENTATION_RESPONSE_SD_THRESHOLD
    ),
    'train_orientation_minimum_eligible_trials': PETERKA_ORIENTATION_MIN_TRIALS,
    'train_responsive_subset_definition': (
        'VISp neurons passing either the Train P1-A or P5-B across-trial mean '
        'response screen; no per-trial response-fraction requirement'
    ),
    'train_full_sequence_alignment_reference': 'measured P1 onset',
    'train_full_sequence_display_signal': (
        'sequence-prebaseline-subtracted denoised fluorescence; '
        'no baseline-SD scaling'
    ),
    'train_ab_responsive_neuron_count': int(len(train_ab_selected)),
    'train_p5_b_responsive_neuron_count': int(len(train_p5_b_selected)),
    'random_sequence_control_available': False,
    'statistical_unit_warning': 'neurons nested within one mouse; not mouse-level inference',
    'signal_warning': 'denoised fluorescence standardized by baseline SD; not paper eFRstd',
    'train_full_sequence_plot_summary': 'raw mean with SEM',
}

if SAVE_RESULTS:
    ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    tables = {
        'data_qc.csv': data_qc,
        'analysis_population_qc.csv': analysis_population_qc,
        'train_peterka_population_by_position.csv': peterka_train_analysis['population_by_position'],
        'train_peterka_neuron_by_position.csv': peterka_train_analysis['neuron_by_position'],
        'train_peterka_position_tests.csv': peterka_train_analysis['position_tests'],
        'train_peterka_p5_directional_test.csv': peterka_train_analysis['p5_directional_test'],
        'train_peterka_p5_spearman_test.csv': peterka_train_analysis['p5_spearman_test'],
        'train_peterka_p5_trajectory.csv': peterka_train_analysis['p5_trajectory'],
        'train_peterka_p5_binned_trajectory.csv': peterka_train_analysis['p5_binned_trajectory'],
        'train_ab_orientation_responsive_neurons.csv': train_ab_neuron_selection,
        'train_p5_b_orientation_responsive_neurons.csv': train_p5_b_neuron_selection,
        'train_sequence_stimulus_timing.csv': train_sequence_stimulus_timing,
        'balanced_test_p5_events.csv': balanced_metadata,
        'test_p5_condition_summary.csv': test_analysis['condition_summary'],
        'test_p5_neuron_condition_response.csv': test_analysis['neuron_condition_response'],
        'test_p5_omnibus_test.csv': test_analysis['omnibus_test'],
        'test_p5_pairwise_tests.csv': test_analysis['pairwise_tests'],
        'literature_proxy_contrasts.csv': proxy_contrasts,
        'predictable_b_equivalence_test.csv': b_equivalence,
        'condition_selection_summary.csv': condition_selection_summary,
        'test_p5_paired_reference_events.csv': p5_paired_events,
        'test_neuron_effects.csv': p5_neuron_effects,
        'test_neuron_effect_quadrants.csv': neuron_effect_quadrants,
        'test_representative_neurons.csv': representative_p5_neurons,
    }
    for symbol, selection in condition_selections.items():
        tables[f'test_p5_{symbol.lower()}_responsive_neurons.csv'] = selection
    for filename, table in tables.items():
        table.to_csv(ANALYSIS_DIR / filename, index=False)

    np.savez_compressed(
        ANALYSIS_DIR / 'matched_p5_aligned_activity.npz',
        cell_index=analysis_cell_indices,
        relative_time_s=balanced_aligned['relative_time_s'],
        window_index=balanced_aligned['window_index'],
        delta_standardized=balanced_aligned_delta_std,
        row_order=matched_heatmap_data['row_order'],
        row_order_cell_index=analysis_cell_indices[matched_heatmap_data['row_order']],
        sort_peak_time_s=matched_row_sort['peak_time_s'],
        sort_peak_amplitude=matched_row_sort['peak_amplitude'],
        color_limit=np.asarray(matched_heatmap_data['color_limit']),
        mean_A=matched_heatmap_data['mean_A'],
        mean_B=matched_heatmap_data['mean_B'],
        mean_C=matched_heatmap_data['mean_C'],
        display_A=matched_heatmap_data['display_A'],
        display_B=matched_heatmap_data['display_B'],
        display_C=matched_heatmap_data['display_C'],
    )
    np.savez_compressed(
        ANALYSIS_DIR / 'train_peterka_sequence_aligned_activity.npz',
        cell_index=train_ab_selected['cell_index'].to_numpy(dtype=np.int64),
        relative_time_s=train_sequence_aligned['relative_time_s'],
        sequence_index=train_sequence_metadata['sequence_index'].to_numpy(dtype=np.int64),
        delta_baseline_subtracted_raw=train_sequence_delta_raw,
        selected_a=train_ab_selected['selected_A'].to_numpy(dtype=bool),
        selected_b=train_ab_selected['selected_B'].to_numpy(dtype=bool),
        population_a_only=train_sequence_population_groups['A-only'],
        population_b_only=train_sequence_population_groups['B-only'],
        population_union=train_sequence_population_groups['A-or-B union'],
        stimulus_item_position=train_sequence_stimulus_timing['item_position'].to_numpy(dtype=np.int64),
        stimulus_symbol=train_sequence_stimulus_timing['symbol'].to_numpy(dtype=str),
        stimulus_relative_onset_s=train_sequence_stimulus_timing['relative_onset_s'].to_numpy(dtype=float),
        stimulus_relative_offset_s=train_sequence_stimulus_timing['relative_offset_s'].to_numpy(dtype=float),
    )
    np.save(ANALYSIS_DIR / 'neuron_baseline_sd_scale.npy', neuron_scale)

    figure_stems = {
        'whole_recording_activity': fig_activity_overview,
        'all_neurons_atlas': fig_all_neurons_atlas,
        'train_peterka_sequence_groups_raw_mean_sem': fig_train_sequence_mean_sem,
        'train_peterka_adaptation_summary': fig_train_summary,
        'test_balanced_p5_traces': fig_test_p5_traces,
        'test_matched_p5_mean_heatmaps': fig_test_p5_heatmaps,
        'test_representative_neuron_traces': fig_representative_traces,
        'test_neuron_effect_scatter': fig_neuron_effect_scatter,
        'test_condition_responsive_atlas': fig_condition_atlas,
    }
    saved_figures = []
    for stem, figure in figure_stems.items():
        saved_figures.extend(save_figure_bundle(figure, FIGURE_DIR / stem))

    with (ANALYSIS_DIR / 'analysis_parameters.json').open('w', encoding='utf-8') as handle:
        json.dump(analysis_parameters, handle, indent=2, ensure_ascii=True)

    p5_test = peterka_train_analysis['p5_directional_test'].iloc[0]
    p5_trend = peterka_train_analysis['p5_spearman_test'].iloc[0]
    proxy_by_name = proxy_contrasts.set_index('comparison')
    a_proxy = proxy_by_name.loc['P5-A < P4-A within AAAAA']
    b_proxy = proxy_by_name.loc['test P5-B vs late-train P5-B']
    c_proxy = proxy_by_name.loc['P5-C > balanced P5-B']
    b_tost = b_equivalence.iloc[0]
    summary = {
        'session_id': SESSION_ID,
        'analysis_region': ANALYSIS_REGION,
        'fixed_neuron_count': int(len(analysis_cell_indices)),
        'train_ab_candidate_visp_neuron_count': int(len(train_ab_neuron_selection)),
        'train_ab_responsive_neuron_count': int(len(train_ab_selected)),
        'train_p1_a_responsive_neuron_count': int(
            train_ab_neuron_selection['selected_A'].sum()
        ),
        'train_p5_b_candidate_visp_neuron_count': int(len(train_p5_b_neuron_selection)),
        'train_p5_b_responsive_neuron_count': int(len(train_p5_b_selected)),
        'train_p5_first_minus_last_mean': float(p5_test['mean_difference']),
        'train_p5_first_minus_last_median': float(p5_test['median_difference']),
        'train_p5_first_greater_last_p': float(p5_test['p_value']),
        'train_p5_direction_supported_at_0_05': bool(p5_test['p_value'] < 0.05),
        'train_p5_median_spearman_rho': float(p5_trend['rho']),
        'train_p5_median_decrease_p': float(p5_trend['p_value_one_sided']),
        'balanced_test_trials': (
            balanced_metadata.groupby('symbol').size().astype(int).to_dict()
        ),
        'p5_a_proxy_mean_difference': float(a_proxy['mean_difference']),
        'p5_a_proxy_holm_p': float(a_proxy['p_holm_3_proxies']),
        'p5_b_proxy_mean_difference': float(b_proxy['mean_difference']),
        'p5_b_proxy_holm_p': float(b_proxy['p_holm_3_proxies']),
        'p5_b_equivalence_margin_sd': float(b_tost['margin']),
        'p5_b_equivalence_p_tost': float(b_tost['p_tost']),
        'p5_b_equivalent_at_0_05': bool(b_tost['equivalent_at_0_05']),
        'p5_c_proxy_mean_difference': float(c_proxy['mean_difference']),
        'p5_c_proxy_holm_p': float(c_proxy['p_holm_3_proxies']),
        'responsive_neuron_counts': {
            row['condition']: int(row['selected_neurons'])
            for row in selection_rows
        },
        'paired_effect_quadrants': (
            neuron_effect_quadrants.set_index('quadrant')['neuron_count']
            .astype(int).to_dict()
        ),
        'representative_neuron_ids': (
            representative_p5_neurons['cell_index'].astype(int).tolist()
        ),
        'proxy_contrasts_are_not_deviant_control_tests': True,
        'random_sequence_control_available': False,
    }
    with (ANALYSIS_DIR / 'analysis_summary.json').open('w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=2, ensure_ascii=True)

    print({
        'saved_directory': ANALYSIS_DIR,
        'saved_tables': len(tables),
        'saved_figures_png_svg': len(saved_figures),
        'next_review': [
            ANALYSIS_DIR / 'analysis_summary.json',
            ANALYSIS_DIR / 'literature_proxy_contrasts.csv',
            FIGURE_DIR / 'train_peterka_sequence_groups_raw_mean_sem.png',
            FIGURE_DIR / 'train_peterka_adaptation_summary.png',
            FIGURE_DIR / 'test_matched_p5_mean_heatmaps.png',
            FIGURE_DIR / 'test_representative_neuron_traces.png',
            FIGURE_DIR / 'test_neuron_effect_scatter.png',
        ],
    })
else:
    print('SAVE_RESULTS=False: plots were displayed but no derived files were written.')

## Interpretation checklist

- **Train full-sequence view:** the main 3 x 2 mean +/- SEM figure aligns every sequence to its measured P1 onset and separately shows A-only, B-only, and their union for first/last sequences. It plots sequence-prebaseline-subtracted denoised fluorescence on one shared y-axis, without baseline-SD scaling.
- **Train inference:** inspect whether first versus last `TRAIN_EDGE_TRIALS` P5-B responses differ (current default: 10) and whether the P5 trajectory decreases. Figure S1 used five bins of seven, 152 responsive neurons from five mice, and a neuron-level mixed model with mouse as a random effect.
- **P5-A proxy:** `P5-A < P4-A` holds orientation and sequence identity fixed, but position/repetition differs.
- **Predictable P5-B proxy:** a non-significant difference does not prove no enhancement. Interpret equivalence only when the TOST passes the pre-specified `B_EQUIVALENCE_MARGIN_SD`.
- **P5-C proxy:** `C > B` remains confounded by orientation because there is no random C control.
- **Heatmaps:** A/B/C use identical neurons, identical row scaling, identical row order, and balanced trials. This corrects the main comparison problem in the former independent heatmaps.
- **Inference boundary:** neuron-level p-values describe this recording. Biological replication requires a mouse-level mixed model after additional sessions are processed.